#Verifying Silver Table

In [0]:
# Step 1: Read existing Silver tables

CATALOG = "retail_analytics_catalog"

SCHEMA = "retail_analytics"

silver_customers = spark.table(

    f"{CATALOG}.{SCHEMA}.silver_customers"

)

silver_orders = spark.table(

    f"{CATALOG}.{SCHEMA}.silver_orders"

)

silver_order_items = spark.table(

    f"{CATALOG}.{SCHEMA}.silver_order_items"

)

silver_products = spark.table(

    f"{CATALOG}.{SCHEMA}.silver_products"

)

print("Silver tables loaded successfully!")

print("Customers:", silver_customers.count())

print("Orders:", silver_orders.count())

print("Order Items:", silver_order_items.count())

print("Products:", silver_products.count())
 

Silver tables loaded successfully!
Customers: 99441
Orders: 99441
Order Items: 112650
Products: 32951


In [0]:
# Step 2: Verify the Gold schema
CATALOG = "retail_analytics_catalog"
SCHEMA = "retail_analytics"
spark.sql(
   f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}_gold"
)
print("Gold schema created or already exists!")
display(
   spark.sql(f"SHOW SCHEMAS IN {CATALOG}")
)

Gold schema created or already exists!


databaseName
default
information_schema
retail_analytics
retail_analytics_gold


i took silver_orders for order id, order purchases timestamp, order status
Silver_order_items - order id, product price, freight value
for revenue calculation need of price and daily/monthly trend  i need order date

for daily sales trend

In [0]:
from pyspark.sql import functions as F
# Step 3: Read Silver tables
orders = spark.table(
   "retail_analytics_catalog.retail_analytics.silver_orders"
)
order_items = spark.table(
   "retail_analytics_catalog.retail_analytics.silver_order_items"
)
# Step 4: Select required columns from orders
valid_orders = orders.select(
   "order_id",
   "order_purchase_timestamp",
   "order_status"
)
# Step 5: Join orders with order items
sales_df = valid_orders.join(
   order_items,
   on="order_id",
   how="inner"
)
# Step 6: Create order date
sales_df = sales_df.withColumn(
   "order_date",
   F.to_date("order_purchase_timestamp")
)
# Step 7: Calculate daily sales metrics
gold_sales_summary = sales_df.groupBy(
   "order_date"
).agg(
   F.sum("price").alias("total_revenue"),
   F.sum("freight_value").alias("total_freight"),
   F.countDistinct("order_id").alias("total_orders"),
   F.countDistinct("order_id").alias("unique_customers")
)
display(gold_sales_summary.orderBy("order_date"))

order_date,total_revenue,total_freight,total_orders,unique_customers
2016-09-04,72.89,63.34,1,1
2016-09-05,59.5,15.56,1,1
2016-09-15,134.97,8.49,1,1
2016-10-02,100.0,9.34,1,1
2016-10-03,463.48,131.66,8,8
2016-10-04,9940.959999999994,1354.5199999999995,60,60
2016-10-05,8343.249999999998,1302.6899999999998,42,42
2016-10-06,7960.509999999994,1170.72,49,49
2016-10-07,7228.0499999999965,907.92,45,45
2016-10-08,8441.849999999997,941.5199999999999,40,40


Correcting daily sales summary


In [0]:
from pyspark.sql import functions as F
# Read the Silver tables
orders = spark.table(
   "retail_analytics_catalog.retail_analytics.silver_orders"
)
order_items = spark.table(
   "retail_analytics_catalog.retail_analytics.silver_order_items"
)
customers = spark.table(
   "retail_analytics_catalog.retail_analytics.silver_customers"
)
# Select required columns
orders_selected = orders.select(
   "order_id",
   "customer_id",
   "order_purchase_timestamp",
   "order_status"
)
customers_selected = customers.select(
   "customer_id",
   "customer_unique_id"
)
# Pre-aggregate order items to order level to avoid countDistinct on order_id
order_item_totals = (
   order_items
   .groupBy("order_id")
   .agg(
       F.sum("price").alias("price"),
       F.sum("freight_value").alias("freight_value")
   )
)
# Join orders with pre-aggregated order items
sales_df = orders_selected.join(
   order_item_totals,
   on="order_id",
   how="inner"
)
# Add actual customer identity
sales_df = sales_df.join(
   customers_selected,
   on="customer_id",
   how="left"
)
# Create order date
sales_df = sales_df.withColumn(
   "order_date",
   F.to_date("order_purchase_timestamp")
)
# Calculate daily sales metrics
gold_sales_summary = (
   sales_df
   .groupBy("order_date")
   .agg(
       F.sum("price").alias("total_revenue"),
       F.sum("freight_value").alias("total_freight"),
       F.count("order_id").alias("total_orders"),
       F.countDistinct("customer_unique_id").alias("unique_customers")
   )
   .withColumn(
       "total_revenue",
       F.round("total_revenue", 2).cast("decimal(18,2)")
   )
   .withColumn(
       "total_freight",
       F.round("total_freight", 2).cast("decimal(18,2)")
   )
   .orderBy("order_date")
)
display(gold_sales_summary)

order_date,total_revenue,total_freight,total_orders,unique_customers
2016-09-04,72.89,63.34,1,1
2016-09-05,59.50,15.56,1,1
2016-09-15,134.97,8.49,1,1
2016-10-02,100.00,9.34,1,1
2016-10-03,463.48,131.66,8,8
2016-10-04,9940.96,1354.52,60,60
2016-10-05,8343.25,1302.69,42,42
2016-10-06,7960.51,1170.72,49,49
2016-10-07,7228.05,907.92,45,45
2016-10-08,8441.85,941.52,40,39


#Monthly Sales Trend

In [0]:
gold_monthly_sales = (
   gold_sales_summary
   .withColumn(
       "sales_month",
       F.date_trunc("month", F.col("order_date"))
   )
   .groupBy("sales_month")
   .agg(
       F.sum("total_revenue").alias("total_revenue"),
       F.sum("total_orders").alias("total_orders")
   )
   .orderBy("sales_month")
)
display(gold_monthly_sales)

sales_month,total_revenue,total_orders
2016-09-01T00:00:00.000Z,267.36,3
2016-10-01T00:00:00.000Z,49507.66,308
2016-12-01T00:00:00.000Z,10.90,1
2017-01-01T00:00:00.000Z,120312.87,789
2017-02-01T00:00:00.000Z,247303.02,1733
2017-03-01T00:00:00.000Z,374344.30,2641
2017-04-01T00:00:00.000Z,359927.23,2391
2017-05-01T00:00:00.000Z,506071.14,3660
2017-06-01T00:00:00.000Z,433038.60,3217
2017-07-01T00:00:00.000Z,498031.48,3969


Checking null values in gold sales summary

In [0]:
display(
   gold_sales_summary.filter(
       F.col("order_date").isNull()
   )
)

order_date,total_revenue,total_freight,total_orders,unique_customers


saving gold sales summary

In [0]:
gold_sales_summary.write \
   .format("delta") \
   .mode("overwrite") \
   .saveAsTable(
       "retail_analytics_catalog.retail_analytics_gold.gold_sales_summary"
   )
print("Gold sales summary saved successfully!")

Gold sales summary saved successfully!


In [0]:
%sql
SELECT *
FROM retail_analytics_catalog.retail_analytics_gold.gold_sales_summary
ORDER BY order_date;

order_date,total_revenue,total_freight,total_orders,unique_customers
2016-09-04,72.89,63.34,1,1
2016-09-05,59.50,15.56,1,1
2016-09-15,134.97,8.49,1,1
2016-10-02,100.00,9.34,1,1
2016-10-03,463.48,131.66,8,8
2016-10-04,9940.96,1354.52,60,60
2016-10-05,8343.25,1302.69,42,42
2016-10-06,7960.51,1170.72,49,49
2016-10-07,7228.05,907.92,45,45
2016-10-08,8441.85,941.52,40,39


#Monthly Sales summary

Saving Monthly Table

In [0]:
gold_monthly_sales.write \
   .format("delta") \
   .mode("overwrite") \
   .saveAsTable(
       "retail_analytics_catalog.retail_analytics_gold.gold_monthly_sales"
   )
print("Gold monthly sales table saved successfully!")

Gold monthly sales table saved successfully!


Gold_Product_Performance

In [0]:
# Step 5: Inspect Silver table columns

print("ORDER ITEMS COLUMNS")

print(silver_order_items.columns)

print("\nPRODUCT COLUMNS")

print(silver_products.columns)
 

ORDER ITEMS COLUMNS
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', '_rescued_data']

PRODUCT COLUMNS
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', '_rescued_data']


Joining & aggregating Product data

In [0]:
from pyspark.sql import functions as F

# Step 5.1: Join order items with products

product_sales = silver_order_items.join(

    silver_products.select(

        "product_id",

        "product_category_name"

    ),

    on="product_id",

    how="left"

)

# Step 5.2: Calculate product-level metrics

gold_product_performance = (

    product_sales

    .groupBy("product_id", "product_category_name")

    .agg(

        F.countDistinct("order_id").alias("total_orders"),

        F.count("order_item_id").alias("items_sold"),

        F.sum("price").alias("total_revenue"),

        F.sum("freight_value").alias("total_freight")

    )

    .withColumn(

        "total_revenue",

        F.round("total_revenue", 2).cast("decimal(18,2)")

    )

    .withColumn(

        "total_freight",

        F.round("total_freight", 2).cast("decimal(18,2)")

    )

    .orderBy(F.desc("total_revenue"))

)

display(gold_product_performance)
 

product_id,product_category_name,total_orders,items_sold,total_revenue,total_freight
bb50f2e236e5eea0100680137654686c,beleza_saude,187,195,63885.00,3721.10
6cdd53843498f92890544667809f1595,beleza_saude,151,156,54730.20,4363.79
d6160fb7873f184099d9bc95e30376af,pcs,35,35,48899.34,1426.84
d1c427060a0f73f6b889a5c7c61f2ac4,informatica_acessorios,323,343,47214.51,13761.52
99a4788cb24856965c36a24e339b6058,cama_mesa_banho,467,488,43025.56,8046.04
3dd2a17168ec895c781a9191c1e95ad7,informatica_acessorios,255,274,41082.60,7129.62
25c38557cf793876c5abdd5931f922db,bebes,38,38,38907.32,1404.63
5f504b3a1c75b73d6151be81eb05bdc9,cool_stuff,63,63,37733.90,3991.91
53b36df67ebb7c41585e8d54d6772e08,relogios_presentes,306,323,37683.42,2274.51
aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,431,527,37608.90,7211.86


In [0]:
gold_product_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "retail_analytics_catalog.retail_analytics_gold.gold_product_performance"
    )

print("Gold product performance table saved successfully!")
 

Gold product performance table saved successfully!


gold_repeat_customers 

In [0]:
print("ORDERS COLUMNS:")
print(silver_orders.columns)
print("CUSTOMERS COLUMNS:")
print(silver_customers.columns)

ORDERS COLUMNS:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', '_rescued_data']
CUSTOMERS COLUMNS:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', '_rescued_data']


In [0]:
from pyspark.sql import functions as F
# Step 1: Join orders with customer details
customer_orders = (
   silver_orders
   .join(
       silver_customers.select(
           "customer_id",
           "customer_unique_id"
       ),
       on="customer_id",
       how="inner"
   )
)
# Step 2: Calculate customer-level metrics
gold_repeat_customers = (
   customer_orders
   .groupBy("customer_unique_id")
   .agg(
       F.countDistinct("order_id").alias("total_orders"),
       F.min("order_purchase_timestamp").alias("first_order_date"),
       F.max("order_purchase_timestamp").alias("last_order_date")
   )
   # Step 3: Keep repeat customers only
   .filter(F.col("total_orders") >= 2)
)
display(gold_repeat_customers)

customer_unique_id,total_orders,first_order_date,last_order_date
7973a6ba9c81ecaeb3d628c33c7c7c48,2,2017-09-19T22:17:14.000Z,2017-09-19T22:17:15.000Z
a97d43b24d76819903192611aca17914,2,2017-12-17T19:40:26.000Z,2017-12-29T19:52:03.000Z
345759b8cb3d30586551de1ca6905df0,2,2017-07-20T14:56:52.000Z,2018-07-03T18:36:57.000Z
d132b863416f85f2abb1a988ca05dd12,3,2017-08-14T13:42:08.000Z,2018-07-21T19:57:05.000Z
453c2895f29b6d9a47f8cb48e00b024b,2,2017-09-18T14:31:42.000Z,2018-04-19T18:41:17.000Z
1605a6c5d93d3488fb621c5323930795,2,2017-03-21T09:21:54.000Z,2017-03-21T09:21:54.000Z
21de770aa4f3a60388b97b416ad38343,2,2018-04-27T17:01:15.000Z,2018-05-01T10:31:57.000Z
72e79d08a90aa4e10f7e5aa76e65b258,2,2017-11-05T15:57:06.000Z,2017-11-05T15:57:07.000Z
b2664990a1ac9e250d98ff5c615c3276,2,2017-08-22T20:12:44.000Z,2017-08-22T20:12:47.000Z
bb656eab01a34ac497457c5a92f5bd07,2,2017-05-12T17:46:25.000Z,2017-09-06T17:41:59.000Z


In [0]:
gold_repeat_customers.write \
   .format("delta") \
   .mode("overwrite") \
   .saveAsTable(
       "retail_analytics_catalog.retail_analytics_gold.gold_repeat_customers"
   )
print("Gold repeat customers table saved successfully!")

Gold repeat customers table saved successfully!


 Average Order Value)

Creating order level revenue

In [0]:
from pyspark.sql import functions as F
# Calculate revenue for each order
order_revenue = (
   silver_order_items
   .groupBy("order_id")
   .agg(
       F.sum("price").alias("order_revenue")
   )
)
display(order_revenue)

order_id,order_revenue
0010b2e5201cc5f1ae7e9c6cc8f5bd00,48.9
0028de0ca693a1bb26448916a81105cc,29.99
0032d07457ae9c806c79368d7d9ce96b,159.0
00471463a6106056c1a2a809f70de640,179.99
005d9a5423d47281ac463a968b3936fb,99.97
0068e836900867da359bd81db9227a33,96.99
006f7dfffe2d90809598e8f1972b829b,39.85
008819c6d5f6da6fa5cd9d50ca927cf4,55.0
00891ba5de66f55000ee358ceea9b345,133.0
00949655cdd8d0465e433a5e6c9643e5,18.0


Repeat customers revenue and ADD AOV

In [0]:
# Join orders with their revenue
customer_order_revenue = (
   customer_orders
   .join(order_revenue, on="order_id", how="inner")
)
# Calculate repeat-customer metrics
gold_repeat_customers = (
   customer_order_revenue
   .groupBy("customer_unique_id")
   .agg(
       F.countDistinct("order_id").alias("total_orders"),
       F.sum("order_revenue").alias("total_revenue"),
       F.min("order_purchase_timestamp").alias("first_order_date"),
       F.max("order_purchase_timestamp").alias("last_order_date")
   )
   .filter(F.col("total_orders") >= 2)
   .withColumn(
       "average_order_value",
       F.round(
           F.col("total_revenue") / F.col("total_orders"), 2
       )
   )
)
display(gold_repeat_customers)

customer_unique_id,total_orders,total_revenue,first_order_date,last_order_date,average_order_value
730e4b7ede33b484516435d39ab4dc49,2,339.98,2017-02-14T10:44:33.000Z,2017-06-07T00:19:14.000Z,169.99
5ad84888519e10ff52d8af00641d41bc,2,465.8,2018-06-01T15:08:12.000Z,2018-08-03T13:42:20.000Z,232.9
cbe71e4cb0b82b97bbec5cd51e0fa3ea,2,205.70000000000002,2018-01-20T00:24:23.000Z,2018-03-04T15:01:18.000Z,102.85
6292600f71fe592f716a3e542f184d18,2,79.2,2017-08-10T18:04:32.000Z,2017-08-10T18:04:32.000Z,39.6
453c2895f29b6d9a47f8cb48e00b024b,2,136.25,2017-09-18T14:31:42.000Z,2018-04-19T18:41:17.000Z,68.13
eed42da7ee2f3ce26ab52758e63d855f,2,113.97999999999999,2018-02-22T23:33:58.000Z,2018-02-22T23:33:59.000Z,56.99
35c62cb24d4c38a9d121fed1a43b2237,2,99.8,2018-07-30T16:45:12.000Z,2018-08-24T12:41:33.000Z,49.9
601bf929b8d0a8feecab068a2110ae1f,2,118.9,2017-12-17T14:36:14.000Z,2017-12-17T15:34:23.000Z,59.45
1605a6c5d93d3488fb621c5323930795,2,492.7,2017-03-21T09:21:54.000Z,2017-03-21T09:21:54.000Z,246.35
c4536b58064f144d0538c9cc69c2765d,2,474.9,2017-10-19T15:53:18.000Z,2017-10-19T15:53:20.000Z,237.45


In [0]:
gold_repeat_customers.write \
   .format("delta") \
   .mode("overwrite") \
   .option("overwriteSchema", "true") \
   .saveAsTable(
       "retail_analytics_catalog.retail_analytics_gold.gold_repeat_customers"
   )
print("Repeat customer metrics updated successfully!")

Repeat customer metrics updated successfully!


In [0]:
%sql
SELECT *
FROM retail_analytics_catalog.retail_analytics_gold.gold_repeat_customers
ORDER BY total_revenue DESC
LIMIT 10;

customer_unique_id,total_orders,total_revenue,first_order_date,last_order_date,average_order_value
da122df9eeddfedc1dc1f5349a1a690c,2,7388.0,2017-04-01T15:58:40.000Z,2017-04-01T15:58:41.000Z,3694.0
c8460e4251689ba205045f3ea17884a1,4,4080.0,2018-08-07T09:03:02.000Z,2018-08-08T14:27:15.000Z,1020.0
055ec572ac7f3c7bdd04a183830ebe59,2,3999.98,2017-06-18T17:22:09.000Z,2017-07-18T18:49:15.000Z,1999.99
59d66d72939bc9497e19d89c61a96d5f,2,3459.0,2017-03-02T12:13:18.000Z,2017-08-10T22:09:50.000Z,1729.5
58c1b085b54c03a1f1ab5f13d64c2b1c,2,2999.98,2017-09-16T12:37:51.000Z,2017-09-17T08:33:40.000Z,1499.99
d77aa95864ae5b42160937615628723a,2,2400.0,2017-08-29T22:35:42.000Z,2017-08-29T22:38:40.000Z,1200.0
7b0eaf68a16e4808e5388c67345033c9,2,2238.42,2018-05-19T11:16:55.000Z,2018-05-19T11:30:35.000Z,1119.21
86df00dc5fd68f4dd5d5945ca19f3ed6,3,2200.0,2017-06-08T10:49:42.000Z,2017-10-13T16:10:18.000Z,733.33
6ddbc64bd04d40f7768ff088d94cbeb8,2,2108.0,2018-04-09T22:58:05.000Z,2018-04-09T22:58:05.000Z,1054.0
73601b1eec55943e90ce8d61253d5c09,2,2009.98,2017-09-26T17:29:19.000Z,2017-11-01T13:39:49.000Z,1004.99


In [0]:
%sql
DESCRIBE TABLE
retail_analytics_catalog.retail_analytics_gold.gold_repeat_customers;


col_name,data_type,comment
customer_unique_id,string,null
total_orders,bigint,null
total_revenue,double,null
first_order_date,timestamp,null
last_order_date,timestamp,null
average_order_value,double,null


Final Gold Tables Validation

In [0]:
%sql
SHOW TABLES IN retail_analytics_catalog.retail_analytics_gold;

database,tableName,isTemporary
retail_analytics_gold,gold_monthly_sales,false
retail_analytics_gold,gold_product_performance,false
retail_analytics_gold,gold_repeat_customers,false
retail_analytics_gold,gold_sales_summary,false


Gold Layer final Validation

In [0]:
%sql

SELECT
    customer_unique_id,
    total_orders,
    total_revenue,
    average_order_value,
    ROUND(total_revenue / total_orders, 2) AS expected_aov
FROM retail_analytics_catalog.retail_analytics_gold.gold_repeat_customers

LIMIT 10;
 

customer_unique_id,total_orders,total_revenue,average_order_value,expected_aov
730e4b7ede33b484516435d39ab4dc49,2,339.98,169.99,169.99
5ad84888519e10ff52d8af00641d41bc,2,465.8,232.9,232.9
cbe71e4cb0b82b97bbec5cd51e0fa3ea,2,205.70000000000002,102.85,102.85
6292600f71fe592f716a3e542f184d18,2,79.2,39.6,39.6
453c2895f29b6d9a47f8cb48e00b024b,2,136.25,68.13,68.13
eed42da7ee2f3ce26ab52758e63d855f,2,113.97999999999999,56.99,56.99
35c62cb24d4c38a9d121fed1a43b2237,2,99.8,49.9,49.9
601bf929b8d0a8feecab068a2110ae1f,2,118.9,59.45,59.45
1605a6c5d93d3488fb621c5323930795,2,492.7,246.35,246.35
c4536b58064f144d0538c9cc69c2765d,2,474.9,237.45,237.45


Gold Tables Record

In [0]:
%sql
SELECT 'Daily Sales' AS table_name, COUNT(*) AS row_count
FROM retail_analytics_catalog.retail_analytics_gold.gold_sales_summary
UNION ALL
SELECT 'Monthly Sales', COUNT(*)
FROM retail_analytics_catalog.retail_analytics_gold.gold_monthly_sales
UNION ALL
SELECT 'Product Performance', COUNT(*)
FROM retail_analytics_catalog.retail_analytics_gold.gold_product_performance
UNION ALL
SELECT 'Repeat Customers', COUNT(*)
FROM retail_analytics_catalog.retail_analytics_gold.gold_repeat_customers;

table_name,row_count
Daily Sales,616
Monthly Sales,24
Product Performance,32951
Repeat Customers,2913


In [0]:
%sql
SELECT COUNT(*) AS mismatched_records
FROM retail_analytics_catalog.retail_analytics_gold.gold_repeat_customers
WHERE ABS(
   average_order_value -
   ROUND(total_revenue / total_orders, 2)
) > 0.01;

mismatched_records
0


Daily Sales Vs Monthly Sales revenue reconciliation

In [0]:
%sql
SELECT
   'Daily Sales' AS table_name,
   ROUND(SUM(total_revenue), 2) AS total_revenue
FROM retail_analytics_catalog.retail_analytics_gold.gold_sales_summary
UNION ALL
SELECT
   'Monthly Sales' AS table_name,
   ROUND(SUM(total_revenue), 2) AS total_revenue
FROM retail_analytics_catalog.retail_analytics_gold.gold_monthly_sales;

table_name,total_revenue
Daily Sales,13591643.70
Monthly Sales,13591643.70


Checking null days

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END)
        AS null_dates
FROM retail_analytics_catalog.retail_analytics_gold.gold_sales_summary;
 

total_rows,null_dates
616,0


checking null months

In [0]:
%sql
SELECT
   COUNT(*) AS total_rows,
   SUM(
       CASE WHEN sales_month IS NULL THEN 1 ELSE 0 END
   ) AS null_months
FROM retail_analytics_catalog.retail_analytics_gold.gold_monthly_sales;

total_rows,null_months
24,0


Product Performance Validation/ checking null in product performance

In [0]:
%sql
SELECT
   COUNT(*) AS total_rows,
   COUNT(DISTINCT product_id) AS unique_products,
   SUM(
       CASE WHEN product_id IS NULL THEN 1 ELSE 0 END
   ) AS null_product_ids
FROM retail_analytics_catalog.retail_analytics_gold.gold_product_performance;

total_rows,unique_products,null_product_ids
32951,32951,0


checking nulls in Repeat customer validation

In [0]:
%sql
SELECT
   COUNT(*) AS total_rows,
   COUNT(DISTINCT customer_unique_id) AS unique_customers,
   SUM(
       CASE WHEN customer_unique_id IS NULL THEN 1 ELSE 0 END
   ) AS null_customer_ids
FROM retail_analytics_catalog.retail_analytics_gold.gold_repeat_customers;

total_rows,unique_customers,null_customer_ids
2913,2913,0


(checking nulls) Validating Gold Sales Summary

In [0]:
%sql
SELECT
   COUNT(*) AS total_rows,
   SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END) AS null_dates,
   SUM(CASE WHEN total_revenue IS NULL THEN 1 ELSE 0 END) AS null_revenue
FROM retail_analytics_catalog.retail_analytics_gold.gold_sales_summary;

total_rows,null_dates,null_revenue
616,0,0


Checking Revenue 
Consistency

In [0]:
%sql
SELECT
   COUNT(*) AS total_rows,
   COUNT(DISTINCT sales_month) AS unique_months,
   MIN(sales_month) AS first_month,
   MAX(sales_month) AS last_month
FROM retail_analytics_catalog.retail_analytics_gold.gold_monthly_sales;

total_rows,unique_months,first_month,last_month
24,24,2016-09-01T00:00:00.000Z,2018-09-01T00:00:00.000Z


Validating Product Performance Revenue with Sales Summary

In [0]:
%sql
SELECT
   'Sales Summary' AS table_name,
   ROUND(SUM(total_revenue), 2) AS total_revenue
FROM retail_analytics_catalog.retail_analytics_gold.gold_sales_summary
UNION ALL
SELECT
   'Product Performance' AS table_name,
   ROUND(SUM(total_revenue), 2) AS total_revenue
FROM retail_analytics_catalog.retail_analytics_gold.gold_product_performance;

table_name,total_revenue
Sales Summary,13591643.70
Product Performance,13591643.70


Validating Repeat Customer Revenue

In [0]:
%sql
SELECT
   COUNT(*) AS mismatched_records
FROM retail_analytics_catalog.retail_analytics_gold.gold_repeat_customers
WHERE ABS(
   average_order_value -
   ROUND(total_revenue / total_orders, 2)
) > 0.01;

mismatched_records
0


Gold Layer Table Verification

In [0]:
%sql
SHOW TABLES IN retail_analytics_catalog.retail_analytics_gold;

database,tableName,isTemporary
retail_analytics_gold,gold_monthly_sales,false
retail_analytics_gold,gold_product_performance,false
retail_analytics_gold,gold_repeat_customers,false
retail_analytics_gold,gold_sales_summary,false
